# Join City Hourly Demand with Weather Data

This notebook joins the city-level hourly bike demand dataset with hourly Chicago weather data using date and hour as join keys.

The output is a combined dataset that can be used for downstream feature engineering and demand analysis.

In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("D:/Projects/chicago-bike-demand")
PROCESSED_DIR = BASE_DIR / "01_data" / "Processed"

demand_path = PROCESSED_DIR / "city_hourly_demand.csv"
weather_path = PROCESSED_DIR / "weather_hourly.csv"
output_path = PROCESSED_DIR / "city_hourly_demand_weather.csv"

print("Demand path:", demand_path)
print("Weather path:", weather_path)
print("Output path:", output_path)

Demand path: D:\Projects\chicago-bike-demand\01_data\Processed\city_hourly_demand.csv
Weather path: D:\Projects\chicago-bike-demand\01_data\Processed\weather_hourly.csv
Output path: D:\Projects\chicago-bike-demand\01_data\Processed\city_hourly_demand_weather.csv


## Load demand and weather data

In [2]:
demand = pd.read_csv(demand_path)
weather = pd.read_csv(weather_path)

print("Demand shape:", demand.shape)
print("Weather shape:", weather.shape)

Demand shape: (2209, 8)
Weather shape: (2232, 6)


## Inspect input datasets

In [3]:
print("Demand columns:")
print(demand.columns.tolist())

print("\nWeather columns:")
print(weather.columns.tolist())

print("\nDemand preview:")
print(demand.head())

print("\nWeather preview:")
print(weather.head())

Demand columns:
['date', 'hour', 'trip_count', 'member_trip_count', 'casual_trip_count', 'avg_trip_duration_min', 'is_weekend', 'rush_hour']

Weather columns:
['time', 'temperature', 'precipitation_mm', 'wind_speed', 'date', 'hour']

Demand preview:
         date  hour  trip_count  member_trip_count  casual_trip_count  \
0  2025-02-28    22           1                  1                  0   
1  2025-02-28    23          25                 19                  6   
2  2025-03-01     0         122                 69                 53   
3  2025-03-01     1          85                 52                 33   
4  2025-03-01     2          46                 26                 20   

   avg_trip_duration_min  is_weekend  rush_hour  
0                 109.84       False      False  
1                  18.15       False      False  
2                   8.53        True      False  
3                   8.42        True      False  
4                   7.60        True      False  

Weather pr

## Align join keys

In [4]:
demand["date"] = pd.to_datetime(demand["date"]).dt.date
weather["date"] = pd.to_datetime(weather["date"]).dt.date

demand["hour"] = demand["hour"].astype(int)
weather["hour"] = weather["hour"].astype(int)

print("Demand date range:", demand["date"].min(), "to", demand["date"].max())
print("Weather date range:", weather["date"].min(), "to", weather["date"].max())

Demand date range: 2025-02-28 to 2025-05-31
Weather date range: 2025-02-28 to 2025-05-31


## Join weather data to hourly demand

In [5]:
merged_df = demand.merge(
    weather[["date", "hour", "temperature", "precipitation_mm", "wind_speed"]],
    on=["date", "hour"],
    how="left"
)

print("Merged shape:", merged_df.shape)
merged_df.head()

Merged shape: (2209, 11)


,date,hour,trip_count,member_trip_count,casual_trip_count,avg_trip_duration_min,is_weekend,rush_hour,temperature,precipitation_mm,wind_speed
0,2025-02-28,22,1,1,0,109.84,False,False,4.4,0.0,21.4
1,2025-02-28,23,25,19,6,18.15,False,False,2.9,0.0,19.5
2,2025-03-01,0,122,69,53,8.53,True,False,1.9,0.0,18.3
3,2025-03-01,1,85,52,33,8.42,True,False,0.8,0.0,17.2
4,2025-03-01,2,46,26,20,7.60,True,False,-0.2,0.0,22.1


## Validate merged dataset

In [6]:
print("Missing values by column:")
print(merged_df.isna().sum())

print("\nDuplicate date-hour rows:")
print(merged_df[["date", "hour"]].duplicated().sum())

print("\nDate range:")
print("Min date:", merged_df["date"].min())
print("Max date:", merged_df["date"].max())

print("\nHour range check:")
print(merged_df["hour"].between(0, 23).all())

Missing values by column:
date                     0
hour                     0
trip_count               0
member_trip_count        0
casual_trip_count        0
avg_trip_duration_min    0
is_weekend               0
rush_hour                0
temperature              0
precipitation_mm         0
wind_speed               0
dtype: int64

Duplicate date-hour rows:
0

Date range:
Min date: 2025-02-28
Max date: 2025-05-31

Hour range check:
True


In [8]:
print("Rows missing weather data:")
print(
    merged_df[["temperature", "precipitation_mm", "wind_speed"]].isna().any(axis=1).sum()
)

Rows missing weather data:
0


## Save merged dataset

In [9]:
output_path.parent.mkdir(parents=True, exist_ok=True)

merged_df.to_csv(output_path, index=False)

print("Merged dataset saved")
print(output_path)

Merged dataset saved
D:\Projects\chicago-bike-demand\01_data\Processed\city_hourly_demand_weather.csv


## Final output check

In [10]:
check_merged = pd.read_csv(output_path)

print("Saved file shape:", check_merged.shape)
check_merged.head()

Saved file shape: (2209, 11)


,date,hour,trip_count,member_trip_count,casual_trip_count,avg_trip_duration_min,is_weekend,rush_hour,temperature,precipitation_mm,wind_speed
0,2025-02-28,22,1,1,0,109.84,False,False,4.4,0.0,21.4
1,2025-02-28,23,25,19,6,18.15,False,False,2.9,0.0,19.5
2,2025-03-01,0,122,69,53,8.53,True,False,1.9,0.0,18.3
3,2025-03-01,1,85,52,33,8.42,True,False,0.8,0.0,17.2
4,2025-03-01,2,46,26,20,7.60,True,False,-0.2,0.0,22.1


## Join Summary

The city-level hourly demand dataset was successfully joined with hourly weather data using `date` and `hour` as composite keys.

The final output includes hourly trip demand metrics alongside temperature, precipitation, and wind speed, and is ready for downstream feature engineering and business analysis.